In [1]:
import pandas as pd
import numpy as np

readmissions = pd.read_csv(
    'FY_2026_Hospital_Readmissions_Reduction_Program_Hospital.csv',
    dtype={'Facility ID': str},
    low_memory=False
)

print(f"Raw shape: {readmissions.shape}")
readmissions.head()

Raw shape: (18330, 12)


,Facility Name,Facility ID,State,Measure Name,Number of Discharges,Footnote,Excess Readmission Ratio,Predicted Readmission Rate,Expected Readmission Rate,Number of Readmissions,Start Date,End Date
0,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-HIP-KNEE-HRRP,NaN,NaN,0.9875,4.5734,4.6311,Too Few to Report,07/01/2021,06/30/2024
1,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-CABG-HRRP,137.0,NaN,0.9531,10.3960,10.9078,13,07/01/2021,06/30/2024
2,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-AMI-HRRP,273.0,NaN,0.9370,13.2998,14.1948,33,07/01/2021,06/30/2024
3,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-COPD-HRRP,122.0,NaN,0.9823,16.6384,16.9389,19,07/01/2021,06/30/2024
4,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-PN-HRRP,507.0,NaN,0.9871,15.7529,15.9591,79,07/01/2021,06/30/2024


In [2]:
print("--- Raw nulls ---")
print(readmissions.isnull().sum())

print("\n--- Raw dtypes ---")
print(readmissions.dtypes)

--- Raw nulls ---
Facility Name                     0
Facility ID                       0
State                             0
Measure Name                      0
Number of Discharges          10088
Footnote                      11343
Excess Readmission Ratio       6610
Predicted Readmission Rate     6610
Expected Readmission Rate      6610
Number of Readmissions         6610
Start Date                        0
End Date                          0
dtype: int64

--- Raw dtypes ---
Facility Name                  object
Facility ID                    object
State                          object
Measure Name                   object
Number of Discharges          float64
Footnote                      float64
Excess Readmission Ratio      float64
Predicted Readmission Rate    float64
Expected Readmission Rate     float64
Number of Readmissions         object
Start Date                     object
End Date                       object
dtype: object


In [3]:
readmissions.columns = (
    readmissions.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.replace(r'[^\w]', '_', regex=True)
)

print(readmissions.columns.tolist())

['facility_name', 'facility_id', 'state', 'measure_name', 'number_of_discharges', 'footnote', 'excess_readmission_ratio', 'predicted_readmission_rate', 'expected_readmission_rate', 'number_of_readmissions', 'start_date', 'end_date']


In [4]:
CMS_NULLS = {
    'Not Available': np.nan,
    'Not Applicable': np.nan,
    'Too Few to Report': np.nan,
    'N/A': np.nan
}

readmissions = readmissions.replace(CMS_NULLS)

print("CMS placeholders replaced ")
print(f"Nulls after replace:\n{readmissions.isnull().sum()}")

CMS placeholders replaced 
Nulls after replace:
facility_name                     0
facility_id                       0
state                             0
measure_name                      0
number_of_discharges          10088
footnote                      11343
excess_readmission_ratio       6610
predicted_readmission_rate     6610
expected_readmission_rate      6610
number_of_readmissions        10293
start_date                        0
end_date                          0
dtype: int64


In [5]:
readmissions['facility_id'] = (
    readmissions['facility_id']
    .str.strip()
    .str.zfill(6)
)

print("facility_id sample:", readmissions['facility_id'].head().tolist())

facility_id sample: ['010001', '010001', '010001', '010001', '010001']


In [6]:
readmissions['number_of_readmissions'] = pd.to_numeric(readmissions['number_of_readmissions'], errors='coerce')

print("Numeric dtypes after cast:")
print(readmissions['number_of_readmissions'].dtypes)

Numeric dtypes after cast:
float64


In [7]:
readmissions['is_suppressed'] = readmissions['footnote'].notna()

print("Suppression breakdown:")
print(readmissions['is_suppressed'].value_counts())

Suppression breakdown:
is_suppressed
False    11343
True      6987
Name: count, dtype: int64


In [8]:
cols_to_drop = [
    'facility_name', 
    'state', 
    'start_date', 'end_date',  
    'footnote'                 
]

readmissions = readmissions.drop(columns=cols_to_drop, errors='ignore')

print(f"Shape after dropping: {readmissions.shape}")
print(readmissions.columns.tolist())

Shape after dropping: (18330, 8)
['facility_id', 'measure_name', 'number_of_discharges', 'excess_readmission_ratio', 'predicted_readmission_rate', 'expected_readmission_rate', 'number_of_readmissions', 'is_suppressed']


In [9]:
dupes = readmissions.duplicated(
    subset=['facility_id', 'measure_name']
).sum()
print(f"True duplicates: {dupes}")

print(f"\nUnique hospitals: {readmissions['facility_id'].nunique()}")
print(f"Unique measures : {readmissions['measure_name'].nunique()}")
print(f"\nMeasures available:")
print(readmissions['measure_name'].value_counts())

True duplicates: 0

Unique hospitals: 3055
Unique measures : 6

Measures available:
measure_name
READM-30-HIP-KNEE-HRRP    3055
READM-30-CABG-HRRP        3055
READM-30-AMI-HRRP         3055
READM-30-COPD-HRRP        3055
READM-30-PN-HRRP          3055
READM-30-HF-HRRP          3055
Name: count, dtype: int64


In [10]:
readmissions['measure_name'] = (
    readmissions['measure_name']
    .str.strip()
    .str.lower()
    .str.replace('-', '_')
    .str.replace(r'[^\w]', '_', regex=True)
)

print(readmissions['measure_name'].unique())

['readm_30_hip_knee_hrrp' 'readm_30_cabg_hrrp' 'readm_30_ami_hrrp'
 'readm_30_copd_hrrp' 'readm_30_pn_hrrp' 'readm_30_hf_hrrp']


In [11]:
def pivot_readm(df, value_col, prefix):
    pivoted = df.pivot_table(
        index='facility_id',
        columns='measure_name',
        values=value_col,
        aggfunc='first'
    ).reset_index()
    pivoted.columns = (
        ['facility_id'] +
        [f'readm_{prefix}_' + col for col in pivoted.columns[1:]]
    )
    return pivoted

readm_ratio     = pivot_readm(readmissions, 'excess_readmission_ratio',    'ratio')
readm_predicted = pivot_readm(readmissions, 'predicted_readmission_rate',  'predicted')
readm_expected  = pivot_readm(readmissions, 'expected_readmission_rate',   'expected')
readm_count     = pivot_readm(readmissions, 'number_of_readmissions',      'count')

readmissions_wide = readm_ratio \
    .merge(readm_predicted, on='facility_id', how='left') \
    .merge(readm_expected,  on='facility_id', how='left') \
    .merge(readm_count,     on='facility_id', how='left')

print(f"Wide table shape: {readmissions_wide.shape}")
readmissions_wide.head()

Wide table shape: (2833, 25)


,facility_id,readm_ratio_readm_30_ami_hrrp,readm_ratio_readm_30_cabg_hrrp,readm_ratio_readm_30_copd_hrrp,readm_ratio_readm_30_hf_hrrp,readm_ratio_readm_30_hip_knee_hrrp,readm_ratio_readm_30_pn_hrrp,readm_predicted_readm_30_ami_hrrp,readm_predicted_readm_30_cabg_hrrp,readm_predicted_readm_30_copd_hrrp,...,readm_expected_readm_30_copd_hrrp,readm_expected_readm_30_hf_hrrp,readm_expected_readm_30_hip_knee_hrrp,readm_expected_readm_30_pn_hrrp,readm_count_readm_30_ami_hrrp,readm_count_readm_30_cabg_hrrp,readm_count_readm_30_copd_hrrp,readm_count_readm_30_hf_hrrp,readm_count_readm_30_hip_knee_hrrp,readm_count_readm_30_pn_hrrp
0,010001,0.9370,0.9531,0.9823,1.0233,0.9875,0.9871,13.2998,10.3960,16.6384,...,16.9389,20.1010,4.6311,15.9591,33.0,13.0,19.0,136.0,NaN,79.0
1,010005,NaN,NaN,0.9308,1.0087,0.8602,0.8727,NaN,NaN,16.8541,...,18.1080,20.7700,5.3609,15.2532,NaN,NaN,17.0,35.0,NaN,30.0
2,010006,0.9195,1.0027,1.0520,0.9925,1.0527,0.9819,11.6567,11.1697,19.1045,...,18.1604,19.3517,5.7783,15.9334,29.0,NaN,37.0,88.0,NaN,105.0
3,010007,NaN,NaN,1.0232,1.0620,NaN,1.0345,NaN,NaN,16.5931,...,16.2168,18.5446,NaN,15.7913,NaN,NaN,NaN,12.0,NaN,16.0
4,010008,NaN,NaN,NaN,NaN,NaN,1.0017,NaN,NaN,NaN,...,NaN,NaN,NaN,13.3883,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
readmissions_wide.to_csv('readmissions_cleaned.csv', index=False)
print("Saved as readmissions_cleaned.csv")

Saved as readmissions_cleaned.csv


In [13]:
print(readmissions_wide.columns.tolist())

['facility_id', 'readm_ratio_readm_30_ami_hrrp', 'readm_ratio_readm_30_cabg_hrrp', 'readm_ratio_readm_30_copd_hrrp', 'readm_ratio_readm_30_hf_hrrp', 'readm_ratio_readm_30_hip_knee_hrrp', 'readm_ratio_readm_30_pn_hrrp', 'readm_predicted_readm_30_ami_hrrp', 'readm_predicted_readm_30_cabg_hrrp', 'readm_predicted_readm_30_copd_hrrp', 'readm_predicted_readm_30_hf_hrrp', 'readm_predicted_readm_30_hip_knee_hrrp', 'readm_predicted_readm_30_pn_hrrp', 'readm_expected_readm_30_ami_hrrp', 'readm_expected_readm_30_cabg_hrrp', 'readm_expected_readm_30_copd_hrrp', 'readm_expected_readm_30_hf_hrrp', 'readm_expected_readm_30_hip_knee_hrrp', 'readm_expected_readm_30_pn_hrrp', 'readm_count_readm_30_ami_hrrp', 'readm_count_readm_30_cabg_hrrp', 'readm_count_readm_30_copd_hrrp', 'readm_count_readm_30_hf_hrrp', 'readm_count_readm_30_hip_knee_hrrp', 'readm_count_readm_30_pn_hrrp']
